In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import xarray as xr
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

experimentFolder = r'O:\HybridDune experiment\data RBR, Ossi\netcdf'                   # path where the data is stored

In [ ]:
def movmean(x, N):
    # calculate moving mean. NB: this divides values at the edges by the window length, instead of the available number of values
    y = uniform_filter1d(x, size=N, mode='constant') # for even window: backward avg. So window 2: x_m(i)=[x(i-1)+x(i)]/2. x_m(i=1) = x(i=1)/2

    # compensate edges for number of values, i.e. the truncated window length
    S1 = np.arange(np.ceil(N/2), N)
    S2 = np.ones(len(x)-N+1)*N
    S3 = np.arange(N-1, np.floor(N/2), -1)
    S = np.concatenate((S1, S2, S3)) 
    return y * N / S

In [4]:

instrumentname_all = [ 'ref P_BS2 (Ossi)', 'S2 P_BS2 (Ossi)', 'S3 P_BS2 (Ossi)', 'S4 P_BS2 (Ossi)',  'S1 P_BS1 (Ossi)', 'S2 P_BS1 (Ossi)', 'S3 P_BS1 (Ossi)', 'S4 P_BS1 (Ossi)' ]
offset_all         = [              93757,             84717,             99188,             87040,        96189,                   93167,             83848,             94741 ]; # [Pa] offset to be added to the pressure data, for instrument calibration
offset_rbr_ref = -543
ossi_resolution    = [            10,            1,            1,            1,           10,           10,            1,          10 ]              # resolution of OSSI in Pa


In [5]:
# # CALCULATE ATMOSPHERIC AIR PRESSURE, ...
# Set smoothing window for atmpospheric pressure
t_smooth_air = 10 # [s]     # measured with 8 hz. But p_water and p_air are measured up to 100m apart (and p_air inside). Affected different by wind gusts, so filter out short-term variation

# Open raw data file  of reference sensor -------------------------------------------------------------------
dataFile =r'O:\HybridDune experiment\data RBR, OSSI\netcdf\raw NetCDF\Deployment period 1\Pressure sensor ref P_BS1 (RBR) raw data - period 1.nc'
with xr.open_dataset(dataFile) as ds:
    # Calibrate referense sensor
    ds['p'] = ds['p'] + offset_rbr_ref  # add offset to pressure data, for instrument calibration

# Determine moving average. 
pAir_smooth_8hz = movmean(ds['p'], 8*t_smooth_air) # smooth over 8hz * n seconds

# interpolate to 20hz 
tAir_20hz   = pd.date_range(ds.t.values[0], ds.t.values[-1] , freq='{}s'.format(1 / 20)) # 20z time vector
pAir_smooth_20hz = np.interp(tAir_20hz, ds['t'], pAir_smooth_8hz) # interpolate to 20hz time vector

# Add to dataset
ds_air_20hz = xr.Dataset( # make dataset for 20hz air pressure
    data_vars={'pAir_smooth': (('t',), pAir_smooth_20hz)},
    coords={'t': tAir_20hz} )

In [ ]:
# Instrument to be corrected, filtered
for i in range(1,8):
    instrumentName = instrumentname_all[i]                                                                               # designated name of the instrument
    dataFile = os.path.join(experimentFolder,'raw NetCDF', 'Deployment period 1', 'Pressure sensor ' + instrumentName + ' raw data - period 1.nc')
    with xr.open_dataset(dataFile) as ds0:
        ds0 = ds0.rename({'p': 'p_abs'})   # rename p to p_abs

    # Correct clock drift
    t = ds0.t.values                    # time
    dt = t - t[0]                       # time since instrument start
    t = t[0] + dt * ds0.t_cal_factor.values
    ds_clockdrift = ds0.assign_coords(t=t)        # assign corrected t to database

    # Interpolate pressure/dataset from drifted clock to exactly 20hz, so that datasets with the same coordinates can be subtracted
    t0 = ds.t_installed.values                                      # first full hour that instrument was installed at the beach
    t_end = ds.t_removed.values                                     # last full hour that instrument was installed at the beach
    t_20hz   = pd.date_range(t0, t_end, freq='{}s'.format(1 / 20)) # 20hz time vector
    P_exact20hz = np.interp(t_20hz, ds_clockdrift.t, ds0.p_abs.values)  # interpolate to 20hz time vector
        
    # crop air pressure and instrument dataset to the same time
    ds_air_20hz_crop = ds_air_20hz.sel(t=slice(t0, t_end))          # crop dataset to the time range of interest
    ds0 = ds0.sel(t=slice(t0, t_end))                               # crop dataset to the time range of interest
    
    # Relative pressure: correct the pressure signal with pAir
    ds0['p_rel'] = P_exact20hz + offset_all[i] - ds_air_20hz_crop['pAir_smooth'] # relative pressure. Add calibration offset per instrument (ds_air already calibrated). 

    # Correct negative pressures
    if i != 0: # only for non-reference sensors
        block_mask = ds0['p_rel'] < 0
        ds0['p_rel'] = ds0['p_rel'].where(~block_mask, 0)               # set negative pressures to zero
        
    # Add/replace instrument metadata
    cal_text = '{}'.format(offset_all[i]) + ' Pa added to raw pressure, based on the period 23dec, 19:11 to 19:26, during the calibration test'
    ds0['p_abs'].attrs = {'units': 'Pa', 'long_name': 'Relative pressure', 'comments': 'corrected for air pressure and calibrated','calibration': cal_text}
    cal_text = '{}'.format(offset_all[i]) + ' Pa added to raw pressure of instrument (and ' + '{}'.format(offset_rbr_ref) + ' Pa added to air pressure of sensor refP1), based on the period 23dec, 19:11 to 19:26, during the calibration test'
    ds0['p_rel'].attrs = {'units': 'Pa', 'long_name': 'Relative pressure', 'comments': 'corrected for air pressure and calibrated','calibration': cal_text}
    ds0.attrs['summary'] = 'Hybrid-Dune campaign: pressure corrected for air pressure'
    ds0.attrs['start time'] = pd.to_datetime(ds.t.values[0]).strftime("%d-%b-%Y %H:%M:%S")
    ds0.attrs['end time'] =   pd.to_datetime(ds.t.values[-1]).strftime("%d-%b-%Y %H:%M:%S")
    ds0.t_cal_factor.attrs = {'long_name': 'time dillation factor applied to correct for clock drift', 'comment': 'applied by multiplying time since instrument start (start of raw data series) with t_cal_factor.'}
    ds0.p_abs.attrs['resolution_raw_data'] = '{} Pa'.format(ossi_resolution[i])  # add resolution of OSSI as attribute
    ds0.p_rel.attrs['resolution_raw_data'] = '{} Pa'.format(ossi_resolution[i])

    # Save to netcdf -------------------------------
    ds0.p_abs.values = np.round(ds0.p_abs.values)    # Round pressure to 1 Pa = 0.1 mm
    ds0.p_rel.values = np.round(ds0.p_rel.values)    
    encoding = {var: {"zlib": True, "complevel": 4} for var in list(ds0.data_vars) + list(ds0.coords)}  # Apply deflate compression to all variables and coordinates in netCDF
    ds0.encoding = encoding  # add the encoding to the dataset (not really necessary, but allows retrieval later on)

    if not os.path.isdir(os.path.join(experimentFolder,'QC', 'Deployment period 1')):
        os.mkdir(os.path.join(experimentFolder,'QC', 'Deployment period 1'))
    ncFilePath = os.path.join(experimentFolder, 'QC', 'Deployment period 1',  'Pressure sensor ' + instrumentName + ' QC p_rel - period 1.nc')
    ds0.to_netcdf(ncFilePath, encoding=encoding) # save to netcdf